In [1]:
from pyspark.sql.functions import col, avg, round, when, lit
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, DateType

# Read Silver capacity utilization and server inventory
df_cap = spark.read.format("delta").table("lh_silver.dbo.capacity_utilization")
df_inv = spark.read.format("delta").table("lh_silver.dbo.server_inventory")

# Aggregate average metrics per server
df_health = df_cap.groupBy("server_id").agg(
    round(avg("cpu_pct"), 2).alias("avg_cpu_pct"),
    round(avg("ram_pct"), 2).alias("avg_ram_pct"),
    round(avg("storage_pct"), 2).alias("avg_storage_pct"),
    round(avg("power_watts"), 2).alias("avg_power_watts"),
    round(avg("network_mbps"), 2).alias("avg_network_mbps")
)

# Compute composite health score (lower utilization = healthier)
df_health = df_health.withColumn(
    "health_score",
    round(100 - ((col("avg_cpu_pct") * 0.4) + 
                  (col("avg_ram_pct") * 0.3) + 
                  (col("avg_storage_pct") * 0.3)), 2)
)

# Add health status label based on score thresholds
df_health = df_health.withColumn(
    "health_status",
    when(col("health_score") >= 70, "Healthy")
    .when(col("health_score") >= 50, "Warning")
    .otherwise("Critical")
)

# Join with inventory for full server context
df_health = df_health.join(df_inv, on="server_id", how="left")

# Add decommissioned server SVR-020 manually with explicit schema
# SVR-020 excluded from capacity_utilization — reintroduced here for complete status breakdown
schema = df_health.schema
df_decomm = spark.createDataFrame(
    [("SVR-020", None, None, None, None, None, 0.0, "Decommissioned",
      "RACK-C2", "DAL-DC3", "DB", 64, 512, None, "Decommissioned")],
    schema
)

# Union active servers with decommissioned server
# allowMissingColumns handles any schema differences between the two dataframes
df_health_final = df_health.unionByName(df_decomm, allowMissingColumns=True)

# Write final health scores to Gold layer
df_health_final.write.format("delta").mode("overwrite").saveAsTable("server_health_scores")

print(f"✅ server_health_scores: {df_health_final.count()} rows written to lh_gold")
print(f"   Active/Warning/Critical servers: {df_health.count()}")
print(f"   Decommissioned servers added: 1")

StatementMeta(, 948b9409-1b49-45d3-b9eb-ee30b213c403, 3, Finished, Available, Finished, False)

✅ server_health_scores: 25 rows written to lh_gold
   Active/Warning/Critical servers: 24
   Decommissioned servers added: 1


In [2]:
from pyspark.sql.functions import col, count, round, avg, sum, when

# Read Silver tables
df_inc = spark.read.format("delta").table("lh_silver.dbo.incident_log")
df_inv = spark.read.format("delta").table("lh_silver.dbo.server_inventory")

# Join incidents with server location
df_inc_loc = df_inc.join(df_inv.select("server_id", "data_center_location"), 
                          on="server_id", how="left")

# Aggregate by location and incident type
df_inc_summary = df_inc_loc.groupBy(
    "data_center_location", 
    "incident_type", 
    "severity"
).agg(
    count("incident_id").alias("total_incidents"),
    round(avg("resolution_time_hrs"), 2).alias("avg_resolution_hrs"),
    sum(when(col("resolved") == False, 1).otherwise(0)).alias("open_incidents")
)

# Write to Gold
df_inc_summary.write.format("delta").mode("overwrite").saveAsTable("incident_summary")

print(f"✅ incident_summary: {df_inc_summary.count()} rows written to lh_gold")

StatementMeta(, 948b9409-1b49-45d3-b9eb-ee30b213c403, 4, Finished, Available, Finished, False)

✅ incident_summary: 37 rows written to lh_gold


In [3]:
from pyspark.sql.functions import col, round, avg, sum, when, lit

# Read Silver tables
df_cap = spark.read.format("delta").table("lh_silver.dbo.capacity_utilization")
df_inv = spark.read.format("delta").table("lh_silver.dbo.server_inventory")

# Join capacity with location
df_pue = df_cap.join(df_inv.select("server_id", "data_center_location", "server_type"),
                      on="server_id", how="left")

# Aggregate power metrics by location and date
df_pue = df_pue.groupBy("data_center_location", "timestamp").agg(
    round(sum("power_watts"), 2).alias("total_power_watts"),
    round(avg("cpu_pct"), 2).alias("avg_cpu_pct"),
    round(avg("ram_pct"), 2).alias("avg_ram_pct")
)

# PUE Proxy = Total Power / (Total Power * avg utilization)
# Real PUE = Total Facility Power / IT Equipment Power
# We approximate using power_watts as IT load proxy
df_pue = df_pue.withColumn(
    "pue_proxy",
    round(
        when(col("avg_cpu_pct") >= 85, lit(1.2))  # High utilization = Efficient
        .when(col("avg_cpu_pct") >= 60, lit(1.4))  # Medium = Acceptable
        .otherwise(lit(1.7)),                        # Low utilization = Inefficient (idle servers waste power)
        3
    )
)


# Flag locations exceeding PUE threshold (industry standard < 1.5)
df_pue = df_pue.withColumn(
    "pue_status",
    when(col("pue_proxy") < lit(1.3), "Efficient")
    .when(col("pue_proxy") < lit(1.5), "Acceptable")
    .otherwise("Inefficient")
)

# Write to Gold
df_pue.write.format("delta").mode("overwrite").saveAsTable("pue_metrics")

print(f"✅ pue_metrics: {df_pue.count()} rows written to lh_gold")

StatementMeta(, 948b9409-1b49-45d3-b9eb-ee30b213c403, 5, Finished, Available, Finished, False)

✅ pue_metrics: 18 rows written to lh_gold


In [4]:
from pyspark.sql.functions import col, round, avg, to_date, when

# Read Silver tables
df_cap = spark.read.format("delta").table("lh_silver.dbo.capacity_utilization")
df_weather = spark.read.format("delta").table("lh_silver.dbo.weather_ashburn")
df_inv = spark.read.format("delta").table("lh_silver.dbo.server_inventory")

# Aggregate capacity metrics by date
df_cap_daily = df_cap.withColumn("date", to_date(col("timestamp"))) \
    .groupBy("date").agg(
        round(avg("cpu_pct"), 2).alias("avg_cpu_pct"),
        round(avg("ram_pct"), 2).alias("avg_ram_pct"),
        round(avg("power_watts"), 2).alias("avg_power_watts"),
        round(avg("network_mbps"), 2).alias("avg_network_mbps")
    )

# Join with weather on date
df_corr = df_cap_daily.join(df_weather, on="date", how="inner")

# Add heat stress indicator
df_corr = df_corr.withColumn(
    "heat_stress",
    when(col("temp_max_f") >= 90, "High")
    .when(col("temp_max_f") >= 75, "Moderate")
    .otherwise("Low")
)

# Write to Gold
df_corr.write.format("delta").mode("overwrite").saveAsTable("weather_performance_correlation")

print(f"✅ weather_performance_correlation: {df_corr.count()} rows written to lh_gold")

StatementMeta(, 948b9409-1b49-45d3-b9eb-ee30b213c403, 6, Finished, Available, Finished, False)

✅ weather_performance_correlation: 7 rows written to lh_gold


In [1]:
# Run in any notebook with lh_gold attached
import pandas as pd

tables = ["server_health_scores", "incident_summary", 
          "pue_metrics", "weather_performance_correlation"]

for table in tables:
    df = spark.read.format("delta").table(f"lh_gold.dbo.{table}").toPandas()
    df.to_csv(f"/lakehouse/default/Files/{table}.csv", index=False)
    print(f"✅ {table}.csv exported — {len(df)} rows")

StatementMeta(, b5d35295-7512-457c-add1-0366b2be24f4, 3, Finished, Available, Finished, False)

✅ server_health_scores.csv exported — 25 rows
✅ incident_summary.csv exported — 37 rows
✅ pue_metrics.csv exported — 28 rows
✅ weather_performance_correlation.csv exported — 12 rows


In [1]:
df_anomaly = spark.read.format("delta").table("server_anomaly_scores").toPandas()
df_anomaly.to_csv("/lakehouse/default/Files/server_anomaly_scores.csv", index=False)
print(f"✅ {len(df_anomaly)} rows exported to lh_gold/Files/")

StatementMeta(, fd99bc54-3d01-4530-a38b-99ab5f965a69, 3, Finished, Available, Finished, True)

✅ 151 rows exported to lh_gold/Files/
